# 4.7 Encoding Categorical Variables

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 4: Data Preprocessing & Exploratory Data Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-04-data-preprocessing-and-exploratory-data-analysis/04.07_encoding-categorical-variables.ipynb)

Companion code for Section 4.7 of the book. Run the cells in order from top to bottom.

**Example 4.7.1**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

# ── Label Encoding - ordinal categories only ─────────────────────
education = pd.Series(['HS','BS','MS','PhD','BS','HS','PhD','MS'])

# Manual ordinal mapping - preserves meaningful order
edu_order = {'HS': 0, 'BS': 1, 'MS': 2, 'PhD': 3}
edu_encoded = education.map(edu_order)
print('Education encoding:')
print(pd.DataFrame({'original': education, 'encoded': edu_encoded}))

# ── OrdinalEncoder - for multiple columns ─────────────────────────
df = pd.DataFrame({
    'education': ['HS','BS','MS','PhD','BS'],
    'income_bracket': ['Low','Medium','High','Medium','Low'],
    'rating': ['Poor','Average','Good','Excellent','Good'],
})

enc = OrdinalEncoder(categories=[
    ['HS','BS','MS','PhD'],
    ['Low','Medium','High'],
    ['Poor','Average','Good','Excellent']
])
df_enc = pd.DataFrame(enc.fit_transform(df), columns=df.columns)
print('\nOrdinal encoding:')
print(df_enc)

# ── LabelEncoder - for target variable (y) ───────────────────────
le = LabelEncoder()
y = ['cat','dog','bird','cat','dog','cat']
y_enc = le.fit_transform(y)
print(f'\nClass labels: {le.classes_}')   # ['bird' 'cat' 'dog']
print(f'Encoded:      {y_enc}')           # [1 2 0 1 2 1]
print(f'Decoded:      {le.inverse_transform(y_enc)}')

**Example 4.7.2**

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

df = pd.DataFrame({
    'city':    ['London','New York','London','Tokyo','Paris','New York'],
    'browser': ['Chrome','Safari','Firefox','Chrome','Chrome','Safari'],
    'age':     [25, 32, 28, 45, 31, 27]
})

# ── pd.get_dummies - simple and readable ─────────────────────────
df_dummies = pd.get_dummies(df, columns=['city','browser'], drop_first=False, dtype=int)
print('One-hot encoded (get_dummies):')
print(df_dummies.to_string())

# drop_first=True: drop one category to avoid multicollinearity
# (k-1 dummies fully describe k categories)
df_drop = pd.get_dummies(df, columns=['city','browser'], drop_first=True, dtype=int)
print(f'\nWith drop_first=True: {df_drop.shape[1]} columns (was {df_dummies.shape[1]})')

# ── sklearn OneHotEncoder - integrates into Pipelines ─────────────
enc = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
X_cat = df[['city','browser']]
X_ohe = enc.fit_transform(X_cat)
feature_names = enc.get_feature_names_out(['city','browser'])
print(f'\nScikit-learn OHE features: {feature_names}')

# ── Handling high-cardinality categorical features ────────────────
# A column with 1000 unique values → 1000 new columns - problematic
# Solution 1: group rare categories into 'Other'
cities = pd.Series(['London']*50 + ['NY']*40 + ['Paris']*30 +
                   ['Tokyo']*20 + ['Sydney']*5 + ['Oslo']*2 + ['Bern']*1)

threshold = 10  # Minimum frequency to keep as its own category
freq = cities.value_counts()
cities_grouped = cities.where(cities.isin(freq[freq>=threshold].index), other='Other')
print(f'\nHigh-cardinality handling:')
print(f'Before: {cities.nunique()} unique values')
print(f'After:  {cities_grouped.nunique()} unique values')
print(cities_grouped.value_counts())

**Example 4.7.3**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold

# ── Target Encoding - replace category with mean of target ────────
# Powerful for high-cardinality; but MUST use cross-fold to avoid leakage
df = pd.DataFrame({
    'city':   ['London','NY','London','Tokyo','Paris','NY','London','Tokyo'],
    'income': [65000, 80000, 70000, 55000, 72000, 82000, 68000, 58000],
    'target': [1, 0, 1, 0, 1, 0, 1, 0]
})

# Naive target encoding (risk of leakage - never use this in production)
target_mean = df.groupby('city')['target'].mean()
df['city_naive_te'] = df['city'].map(target_mean)
print('Naive target encoding (leaky!):')
print(df[['city','target','city_naive_te']])

# Cross-fold target encoding (safe - avoids leakage)
def cross_fold_target_encode(df, col, target, n_splits=5, smoothing=10):
    '''Target encoding with k-fold cross-fitting to prevent leakage.'''
    global_mean = df[target].mean()
    encoded = np.zeros(len(df))

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    for train_idx, val_idx in kf.split(df):
        train_fold = df.iloc[train_idx]
        # Compute target mean per category ON TRAINING FOLD ONLY
        means = train_fold.groupby(col)[target].mean()
        # Smoothing: pull rare categories toward global mean
        counts = train_fold.groupby(col)[target].count()
        smoothed = (counts * means + smoothing * global_mean) / (counts + smoothing)
        # Assign to validation fold
        encoded[val_idx] = df.iloc[val_idx][col].map(smoothed).fillna(global_mean)

    return encoded

df['city_safe_te'] = cross_fold_target_encode(df, 'city', 'target')
print('\nSafe cross-fold target encoding:')
print(df[['city','target','city_naive_te','city_safe_te']].round(4))

# ── Summary: when to use each encoding ───────────────────────────
print('\nEncoding selection guide:')
encoding_guide = {
    'Ordinal (ordered categories, low cardinality)':   'OrdinalEncoder with explicit order',
    'Nominal, low cardinality (<20)':                  'OneHotEncoder with drop=first',
    'Nominal, high cardinality (20-1000)':             'TargetEncoder (cross-fold)',
    'Very high cardinality (>1000, e.g. zip codes)':   'Embeddings or frequency encoding',
    'Binary (Yes/No, True/False)':                     'Map to 0/1 directly',
    'Target variable (y)':                             'LabelEncoder',
}
for situation, method in encoding_guide.items():
    print(f'  {situation}')
    print(f'    → {method}')